# Task 3 - Task Automation with Python Scripts

Two small automation scripts: an **email extractor** and a **.jpg file mover**.

**Skills practised:** `re`, `shutil`, `pathlib`/`os`, file handling, `argparse`, error handling.

---
## Module: `email_extractor.py`

### Step 1 - The email pattern
A regular expression describes what an email address looks like: `name` + `@` + `domain` + `.` + `extension`.

In [ ]:
import argparse
import re

EMAIL_PATTERN = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"

### Step 2 - Extract emails from text
`re.findall` returns all matches. `set()` removes duplicates and `sorted()` makes the output tidy.

In [ ]:
def extract_emails(text):
    """Return a sorted list of unique email addresses found in text."""
    return sorted(set(re.findall(EMAIL_PATTERN, text)))

### Step 3 - Read a file, extract, save
A `with` block closes files automatically.

In [ ]:
def extract_from_file(input_path, output_path):
    """Read input_path, save unique emails to output_path, return the list."""
    with open(input_path, encoding="utf-8") as f:
        text = f.read()
    emails = extract_emails(text)
    if emails:
        with open(output_path, "w", encoding="utf-8") as f:
            f.write("\n".join(emails) + "\n")
    return emails

### Step 4 - Command-line interface
`argparse` lets people run `python email_extractor.py sample.txt -o emails.txt`.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Extract email addresses from a text file.")
    parser.add_argument("input", help="Path to the input .txt file")
    parser.add_argument("-o", "--output", default="emails.txt", help="Output file (default: emails.txt)")
    args = parser.parse_args()

    try:
        emails = extract_from_file(args.input, args.output)
    except FileNotFoundError:
        print("Error: file '" + args.input + "' not found.")
        return

    if emails:
        print("Found " + str(len(emails)) + " unique email(s). Saved to " + args.output)
    else:
        print("No email addresses found.")

### Try it on a sample string

In [ ]:
sample = """Contact support@example.com or sales@example.com.
John: john.doe99@mail.co.in, not-an-email@, another@@bad
Duplicate: support@example.com"""
extract_emails(sample)

---
## Module: `jpg_mover.py`

### Step 1 - Imports
`pathlib` handles paths, `shutil.move` moves files.

In [ ]:
import argparse
import shutil
from pathlib import Path

JPG_EXTENSIONS = {".jpg", ".jpeg"}

### Step 2 - Move the files
We loop over the folder, keep only `.jpg` / `.jpeg` files (any letter case), and add a number to the name if a file with the same name already exists, so nothing is overwritten.

In [ ]:
def move_jpgs(source, destination):
    """Move all .jpg/.jpeg files from source to destination. Returns moved file names."""
    source = Path(source)
    destination = Path(destination)
    if not source.is_dir():
        raise FileNotFoundError("Source folder not found: " + str(source))
    destination.mkdir(parents=True, exist_ok=True)

    moved = []
    for file in source.iterdir():
        if file.is_file() and file.suffix.lower() in JPG_EXTENSIONS:
            target = destination / file.name
            counter = 1
            while target.exists():
                target = destination / (file.stem + "_" + str(counter) + file.suffix)
                counter += 1
            shutil.move(str(file), str(target))
            moved.append(target.name)
    return sorted(moved)

### Step 3 - Command-line interface

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Move all .jpg files into a new folder.")
    parser.add_argument("source", help="Folder to scan")
    parser.add_argument("destination", help="Folder to move the images into")
    args = parser.parse_args()

    try:
        moved = move_jpgs(args.source, args.destination)
    except FileNotFoundError as error:
        print("Error:", error)
        return
    print("Moved " + str(len(moved)) + " file(s) to " + args.destination)

### Try it safely with a temporary folder

In [ ]:
import tempfile, os
with tempfile.TemporaryDirectory() as tmp:
    for name in ["a.jpg", "b.JPG", "notes.txt"]:
        open(os.path.join(tmp, name), "w").close()
    print("Moved:", move_jpgs(tmp, os.path.join(tmp, "images")))
    print("Left behind:", sorted(f for f in os.listdir(tmp) if f != "images"))